In [ ]:
!pip install torch==2.1.0+cu121 torchvision==0.16.0+cu121 torchaudio==2.1.0 \
 --extra-index-url https://download.pytorch.org/whl/cu121

Defaulting to user installation because normal site-packages is not writeable


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121


In [ ]:
!pip uninstall numpy -y
!pip install numpy==1.26.4

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.12.0 requires numpy<1.24,>=1.22, but you have numpy 1.26.4 which is incompatible.


In [ ]:
!pip install \
    torch-scatter \
    torch-sparse \
    torch-cluster \
    torch-spline-conv \
    -f https://data.pyg.org/whl/torch-2.1.0+cu121.html


Defaulting to user installation because normal site-packages is not writeable


Looking in links: https://data.pyg.org/whl/torch-2.1.0+cu121.html


In [ ]:
!pip install torch-geometric

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)

2.1.0+cu121
12.1


In [ ]:
# Install dependencies (run once)
!pip install pyvista scikit-learn


Defaulting to user installation because normal site-packages is not writeable


In [ ]:
!pip install trimesh

Defaulting to user installation because normal site-packages is not writeable


## Mesh to Point Cloud conversion

In [ ]:
# ===============================
# STL → Point Cloud Conversion
# ===============================

import os
import numpy as np
import trimesh
from tqdm import tqdm

# -------- Paths --------
STL_DIR = "/home/exouser/02_triangular_mesh_autoencoder/Dataset/F_D_WM_WW_1"
OUT_DIR = "/home/exouser/02_triangular_mesh_autoencoder/Dataset/drivAERnet_PC"

NUM_POINTS = 4096

os.makedirs(OUT_DIR, exist_ok=True)

# -------- Normalization Function --------
def normalize_points(points):
    """
    Center and scale point cloud to unit sphere
    """
    centroid = np.mean(points, axis=0)
    points = points - centroid
    scale = np.max(np.linalg.norm(points, axis=1))
    points = points / scale
    return points

# -------- Main Conversion Loop --------
stl_files = [f for f in os.listdir(STL_DIR) if f.endswith(".stl")]

print(f"Found {len(stl_files)} STL files")

for stl_file in tqdm(stl_files, desc="Converting STL → Point Cloud"):
    stl_path = os.path.join(STL_DIR, stl_file)

    # Load mesh
    mesh = trimesh.load(stl_path, force='mesh')

    if not isinstance(mesh, trimesh.Trimesh):
        print(f"Skipping invalid mesh: {stl_file}")
        continue

    # Sample points uniformly from surface
    points, face_indices = trimesh.sample.sample_surface(mesh, NUM_POINTS)

    # Compute normals
    normals = mesh.face_normals[face_indices]

    # Normalize
    points = normalize_points(points)

    # Save
    out_name = stl_file.replace(".stl", ".npz")
    out_path = os.path.join(OUT_DIR, out_name)

    np.savez(
        out_path,
        points=points.astype(np.float32),
        normals=normals.astype(np.float32)
    )

print("Point cloud generation completed.")


Found 403 STL files


Converting STL → Point Cloud: 100%|██████████| 403/403 [04:00<00:00,  1.67it/s]

Point cloud generation completed.


In [ ]:
!pip install open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 115.1 MB/s  0:00:030:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 144.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 43.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 85.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 135.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 124.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 47.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28/28 [open3d]27/28 [open3d]ets]rse]ion]


In [ ]:
import numpy as np
import open3d as o3d

# --------- CONFIG ---------
npz_path = "/home/exouser/02_triangular_mesh_autoencoder/Dataset/drivAERnet_PC/F_D_WM_WW_0003.npz"
ply_path = "/home/exouser/02_triangular_mesh_autoencoder/output_mesh/output_0003.ply"

# Possible keys commonly used in NPZ point clouds
POINT_KEYS = ["points", "xyz", "vertices", "pcd"]
COLOR_KEYS = ["colors", "rgb", "color"]
# --------------------------

# Load NPZ file
data = np.load(npz_path)

# Find point array
points = None
for k in POINT_KEYS:
    if k in data:
        points = data[k]
        break

if points is None:
    raise KeyError(f"No point array found. Available keys: {list(data.keys())}")

# Find color array (optional)
colors = None
for k in COLOR_KEYS:
    if k in data:
        colors = data[k]
        break

# Ensure correct shapes
points = np.asarray(points).reshape(-1, 3)

if colors is not None:
    colors = np.asarray(colors).reshape(-1, 3)
    # Normalize colors if needed (0–255 → 0–1)
    if colors.max() > 1.0:
        colors = colors / 255.0

# Create Open3D point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

if colors is not None:
    pcd.colors = o3d.utility.Vector3dVector(colors)

# Save to PLY
o3d.io.write_point_cloud(ply_path, pcd)

print(f"Saved {len(points)} points to {ply_path}")


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Saved 4096 points to /home/exouser/02_triangular_mesh_autoencoder/output_mesh/output_0003.ply


In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    total_mem = torch.cuda.get_device_properties(device).total_memory
    allocated = torch.cuda.memory_allocated(device)
    reserved = torch.cuda.memory_reserved(device)

    print(f"GPU: {torch.cuda.get_device_name(device)}")
    print(f"Total memory: {total_mem / 1024**3:.2f} GB")
    print(f"Allocated: {allocated / 1024**3:.2f} GB")
    print(f"Reserved: {reserved / 1024**3:.2f} GB")
else:
    print("CUDA not available")


GPU: GRID A100X-10C
Total memory: 10.00 GB
Allocated: 0.02 GB
Reserved: 1.84 GB


In [ ]:
import subprocess

output = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=memory.total,memory.used,memory.free",
        "--format=csv,nounits,noheader"
    ],
    encoding="utf-8"
)

total, used, free = output.strip().split(", ")

print(f"Total GPU memory: {total} MB")
print(f"Used GPU memory: {used} MB")
print(f"Free GPU memory left: {free} MB")


Total GPU memory: 10240 MB
Used GPU memory: 2410 MB
Free GPU memory left: 6913 MB


In [ ]:
!nvidia-smi

Wed Jan 21 05:14:57 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.247.01             Driver Version: 535.247.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  GRID A100X-10C                 On  | 00000000:04:00.0 Off |                    0 |
| N/A   N/A    P0              N/A /  N/A |   2410MiB / 10240MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

# Drag Coefficient Prediction using RegDGCNN



In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

NUM_POINTS = 4096
BATCH_SIZE = 4       # Safe for ~10GB GPU
EPOCHS = 100
LR = 1e-3
K = 20                # k-NN for EdgeConv


Using device: cuda


In [ ]:
CSV_PATH = "/home/exouser/02_triangular_mesh_autoencoder/Dataset/DrivAerNetPlusPlus_Cd_8k_Updated.csv"
PC_DIR   = "/home/exouser/02_triangular_mesh_autoencoder/Dataset/drivAERnet_PC"

df = pd.read_csv(CSV_PATH)

available_ids = set(
    f.replace(".npz", "")
    for f in os.listdir(PC_DIR)
    if f.endswith(".npz")
)

print("Point clouds available:", len(available_ids))
print("CSV entries before   :", len(df))

df = df[df["ID"].isin(available_ids)].reset_index(drop=True)

print("CSV entries after    :", len(df))


Point clouds available: 500
CSV entries before   : 8121
CSV entries after    : 499


In [ ]:
train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.2,
    random_state=42
)


In [ ]:
class DrivAerPointCloudDataset(Dataset):
    def __init__(self, df, pc_dir, indices):
        self.df = df.iloc[indices].reset_index(drop=True)
        self.pc_dir = pc_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pc_path = os.path.join(self.pc_dir, row["ID"] + ".npz")

        data = np.load(pc_path)
        points  = data["points"]    # (4096, 3)
        normals = data["normals"]   # (4096, 3)

        pc = np.concatenate([points, normals], axis=1)  # (4096, 6)
        pc = torch.tensor(pc, dtype=torch.float32)

        cd = torch.tensor(row["Drag_Value"], dtype=torch.float32)

        return pc, cd


In [ ]:
train_ds = DrivAerPointCloudDataset(df, PC_DIR, train_idx)
test_ds  = DrivAerPointCloudDataset(df, PC_DIR, test_idx)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)

test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False
)


In [ ]:
def knn(x, k):
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
    return pairwise_distance.topk(k=k, dim=-1)[1]


In [ ]:
def get_graph_feature(x, k):
    B, C, N = x.size()
    idx = knn(x, k)
    device = x.device

    idx_base = torch.arange(0, B, device=device).view(-1, 1, 1) * N
    idx = (idx + idx_base).view(-1)

    x = x.transpose(2, 1).contiguous()
    feature = x.view(B * N, -1)[idx, :]
    feature = feature.view(B, N, k, C)

    x = x.view(B, N, 1, C).repeat(1, 1, k, 1)
    feature = torch.cat((feature - x, x), dim=3)

    return feature.permute(0, 3, 1, 2)


In [ ]:
class RegDGCNN(nn.Module):
    def __init__(self, k=40):
        super().__init__()
        self.k = k

        self.conv1 = nn.Sequential(
            nn.Conv2d(12, 64, 1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 128, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2)
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(256, 256, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2)
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(512, 512, 1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2)
        )

        self.fc = nn.Sequential(
            nn.Linear(64 + 128 + 256 + 512, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            nn.Linear(256, 1)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)

        x1 = self.conv1(get_graph_feature(x, self.k)).max(dim=-1)[0]
        x2 = self.conv2(get_graph_feature(x1, self.k)).max(dim=-1)[0]
        x3 = self.conv3(get_graph_feature(x2, self.k)).max(dim=-1)[0]
        x4 = self.conv4(get_graph_feature(x3, self.k)).max(dim=-1)[0]

        x = torch.cat([x1, x2, x3, x4], dim=1)
        x = torch.max(x, dim=2)[0]

        return self.fc(x).squeeze(1)


In [ ]:
model = RegDGCNN(k=K).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()


In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for pc, cd in tqdm(train_loader, leave=False):
        pc, cd = pc.to(DEVICE), cd.to(DEVICE)

        optimizer.zero_grad()
        pred = model(pc)
        loss = criterion(pred, cd)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1:03d} | Train MSE: {total_loss / len(train_loader):.6f}")


Epoch 001 | Train MSE: 0.090329


Epoch 002 | Train MSE: 0.012896


Epoch 003 | Train MSE: 0.004500


Epoch 004 | Train MSE: 0.003141


Epoch 005 | Train MSE: 0.002494


...

Epoch 096 | Train MSE: 0.000279


Epoch 097 | Train MSE: 0.000260


Epoch 098 | Train MSE: 0.000291


Epoch 099 | Train MSE: 0.000318


Epoch 100 | Train MSE: 0.000300


In [ ]:
model.eval()
gt, pred = [], []

with torch.no_grad():
    for pc, cd in test_loader:
        pc = pc.to(DEVICE)
        output = model(pc).cpu().numpy()

        pred.extend(output)
        gt.extend(cd.numpy())


In [ ]:
print("MAE :", mean_absolute_error(gt, pred))
print("RMSE:", np.sqrt(mean_squared_error(gt, pred)))
print("R2  :", r2_score(gt, pred))


MAE : 0.03630456730723381
RMSE: 0.03802151625947023
R2  : -15.009676991507039


In [ ]:
results = pd.DataFrame({
    "GT_Cd": gt,
    "Predicted_Cd": pred
})

results["Error"] = results["Predicted_Cd"] - results["GT_Cd"]
results["Abs_Error"] = results["Error"].abs()

results.sort_values("Abs_Error", ascending=False).head(20)


,GT_Cd,Predicted_Cd,Error,Abs_Error
40,0.303021,0.233651,-0.069370,0.069370
17,0.282586,0.219093,-0.063494,0.063494
67,0.295759,0.232988,-0.062770,0.062770
23,0.272138,0.209824,-0.062314,0.062314
99,0.284060,0.228376,-0.055684,0.055684
84,0.286206,0.231441,-0.054765,0.054765
22,0.301783,0.248724,-0.053059,0.053059
64,0.276101,0.223117,-0.052984,0.052984
4,0.277643,0.225256,-0.052387,0.052387
63,0.270062,0.218033,-0.052029,0.052029
